# Tutorial 2 — Regressão logística versus Random Forest**Nível:** Iniciante/intermediário · **Tempo:** 40 min**Pergunta:** É possível prever um desfecho neonatal raro a partir de variáveis disponíveis no momento do nascimento?> Dados didáticos sintéticos, não constituem evidência empírica.

### Como utilizar este notebook

1. Execute as células **de cima para baixo** usando o botão ▶ do Google Colab.
2. Leia a explicação antes de cada código. Não é necessário decorar comandos.
3. Compare os resultados mostrados na saída com as perguntas de interpretação.
4. Experimente as mudanças sugeridas na seção **Agora é sua vez**.

**Convenções de leitura:** `df` representa uma tabela, `X` reúne variáveis preditoras e `y` representa o resultado esperado, quando aplicável.

> **Importante:** todos os dados deste notebook são sintéticos. Os resultados não devem ser empregados para decisões reais.


## Passo 1 — Bibliotecas

Os modelos são implementados com scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay

## Passo 2 — Simular dados

O desfecho é inteiramente simulado e não representa risco real nem regras clínicas.

**O que observar:** O simulador combina variáveis para gerar probabilidades fictícias. O desfecho é sorteado a partir delas; não são dados do SINASC/SIM.

In [ ]:
rng=np.random.default_rng(7)
n=6000
idade_materna=rng.integers(15, 46, n)
semanas=np.clip(rng.normal(38, 2.4, n), 24, 42)
consultas=rng.integers(0, 12, n)
logit=-3.3+0.13*(37-semanas)+0.025*np.abs(idade_materna-28)-0.055*consultas
prob=1/(1+np.exp(-logit))
obito=rng.binomial(1, prob)
df=pd.DataFrame({"idade_materna":idade_materna, "semanas_gestacao":semanas, "consultas_prenatal":consultas, "obito_neonatal":obito})
print(df.obito_neonatal.value_counts(normalize=True))

## Passo 3 — Separar treino e teste

O conjunto de teste permanece separado durante o ajuste.

**O que observar:** `X` contém os preditores; `y`, o desfecho. A separação estratificada preserva aproximadamente a prevalência nas duas amostras.

In [ ]:
X=df.drop(columns="obito_neonatal")
y=df.obito_neonatal
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=.25, stratify=y, random_state=42)

## Passo 4 — Treinar os dois modelos

Os parâmetros foram escolhidos para a demonstração; pesquisa real exige validação apropriada.

**O que observar:** A regressão passa por padronização; a floresta não exige esse passo. Ambos usam os mesmos registros de treino.

In [ ]:
models={"Regressão logística":make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")), "Random Forest":RandomForestClassifier(n_estimators=150, min_samples_leaf=10, class_weight="balanced", random_state=42, n_jobs=-1)}
for model in models.values(): model.fit(X_train, y_train)

## Passo 5 — Avaliar

Compare a detecção da classe minoritária e a matriz de confusão. AUC e limiar não resolvem questões de calibração.

**O que observar:** A ROC AUC avalia discriminação entre casos; a matriz mostra erros e acertos no limiar escolhido. O limiar de 0,5 não é necessariamente o melhor.

In [ ]:
for nome, model in models.items():
    prob_pred=model.predict_proba(X_test)[:, 1]
    pred=(prob_pred>=0.5).astype(int)
    print("\n", nome, "| ROC AUC:", round(roc_auc_score(y_test, prob_pred), 3))
    print(classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, values_format="d")
    plt.title(nome); plt.show()

## Experimente você mesmoTeste limiares de 0,3 e 0,7 em ambas as previsões. Compare sensibilidade e precisão; explique por que não escolhemos apenas pela acurácia.## Limitações e cuidados metodológicosAs variáveis e o evento foram simulados; modelos não podem orientar decisões clínicas sem validação externa, governança e avaliação de riscos.